In [ ]:
!pip install spotipy

import json
import random
import numpy as np
from datetime import datetime, timedelta
import spotipy
from spotipy.oauth2 import SpotifyOAuth

from dotenv import load_dotenv
import os

load_dotenv()


CLIENT_ID =  os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

# Authenticate with user auth
print("Authenticating...")
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri="http://127.0.0.1:8080",
    scope="playlist-read-private user-read-recently-played user-top-read",
    open_browser=True,
    cache_path=".spotify_cache"
))

# Test connection
user = sp.current_user()
print(f"Logged in as: {user['display_name']}")



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Authenticating...
Logged in as: ୨♡୧


In [ ]:
import json
import random
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import pandas as pd

print("Loading audio features from audio_features_clean.csv...")
audio_df = pd.read_csv("audio_features_clean.csv")
print(f"Loaded {len(audio_df):,} tracks with audio features")

# Create track lookup by emotion
tracks_by_emotion = {}
for _, track in audio_df.iterrows():
    emotion = track["emotion"]
    if emotion not in tracks_by_emotion:
        tracks_by_emotion[emotion] = []
    tracks_by_emotion[emotion].append(track.to_dict())

print(f"\nTracks available per emotion:")
for emotion, tracks in tracks_by_emotion.items():
    print(f"  {emotion}: {len(tracks):,} tracks")


PLAYLISTS_BY_EMOTION = {
    "happy": [
        "Feel-Good Indie Rock", "Happy Hits", "Positive Vibes", "Good Mood Mix",
        "Morning Energy", "Sunshine Sounds", "Upbeat Anthems", "Joyful Tunes",
        "Happy Place", "Smile Mix", "Good Times", "Uplifting Beats",
        "Euphoria", "Pure Joy", "Dancing in the Sun", "Weekend Happiness"
    ],
    "sad": [
        "Sad Songs", "Melancholy Moods", "Heartbreak Hotel", "Emotional Ballads",
        "Rainy Day Mix", "Crying in the Club", "Sad Indie", "Acoustic Sadness",
        "Midnight Thoughts", "Broken Hearts", "Tears & Rain", "Lonely Nights",
        "Dark Academia", "Depressed Vibes", "Heartache", "Sorrowful Sounds"
    ],
    "chill": [
        "Chill Vibes", "Lofi Beats", "Relaxing Music", "Calm Down",
        "Peaceful Moments", "Sunday Morning", "Cozy Evening", "Ambient Waves",
        "Dreamy Escape", "Slow Flow", "Mellow Tunes", "Zen Mode",
        "Midnight Chill", "Cloud Nine", "Hazy Days", "Soft Focus"
    ],
    "energetic": [
        "Workout Mix", "Gym Beats", "Running Songs", "Power Up",
        "High Energy", "Pump Up", "Cardio Blast", "Fitness Motivation",
        "Beast Mode", "Training Session", "Get Moving", "Energy Boost",
        "Maximum Intensity", "Peak Performance", "Hype Train", "Ultimate Workout"
    ],
    "calm": [
        "Meditation", "Sleep Music", "Peaceful Piano", "Ambient Sounds",
        "Deep Relax", "Calm Mind", "Stress Relief", "Gentle Flow",
        "Quiet Time", "Serenity", "Mindful Moments", "Tranquil Dreams",
        "Binaural Beats", "Healing Frequencies", "Nature Sounds", "Deep Sleep"
    ],
    "party": [
        "Party Anthems", "Club Hits", "Dance Party", "Weekend Vibes",
        "Party Starter", "Late Night Mix", "Festival Hits", "Dance Floor",
        "Celebration Time", "Party Rock", "Let's Dance", "Vibes Only",
        "After Party", "Club Bangers", "Mega Mix", "Party All Night"
    ],
    "romantic": [
        "Love Songs", "Romantic Date", "Cuddle Music", "Wedding Songs",
        "R&B Love", "Slow Dance", "Romantic Evening", "Falling in Love",
        "Sweet Moments", "Heartfelt", "Soulmate", "Love Story",
        "First Kiss", "Endless Love", "Valentine's Day", "Passion"
    ],
    "focus": [
        "Study Beats", "Focus Music", "Concentration", "Work Flow",
        "Productivity Mix", "Deep Focus", "Brain Power", "Study Session",
        "Work Mode", "Office Vibes", "Cognitive Flow", "Intense Focus",
        "Programming Mix", "Writing Session", "Exam Prep", "Deep Work"
    ]
}


TIME_PREFERENCES = {
    "sad": {"morning": 0.03, "afternoon": 0.12, "evening": 0.25, "night": 0.60},
    "happy": {"morning": 0.35, "afternoon": 0.40, "evening": 0.18, "night": 0.07},
    "chill": {"morning": 0.40, "afternoon": 0.30, "evening": 0.20, "night": 0.10},
    "energetic": {"morning": 0.25, "afternoon": 0.40, "evening": 0.28, "night": 0.07},
    "calm": {"morning": 0.55, "afternoon": 0.25, "evening": 0.12, "night": 0.08},
    "party": {"morning": 0.03, "afternoon": 0.15, "evening": 0.50, "night": 0.32},
    "romantic": {"morning": 0.08, "afternoon": 0.18, "evening": 0.45, "night": 0.29},
    "focus": {"morning": 0.38, "afternoon": 0.42, "evening": 0.15, "night": 0.05}
}

# Day of week modifiers (weekend vs weekday)
DAY_MODIFIERS = {
    "weekday": {
        "party": 0.3,
        "energetic": 1.2,
        "focus": 1.5,
        "happy": 1.0,
        "sad": 1.0
    },
    "weekend": {
        "party": 2.5,
        "energetic": 1.5,
        "focus": 0.4,
        "happy": 1.3,
        "sad": 0.7,
        "romantic": 1.2,
        "chill": 1.3
    }
}

ANOMALY_TYPES = {
    "late_night_sad": {"weight": 0.20, "description": "Listening to sad music late at night"},
    "sudden_emotion_shift": {"weight": 0.15, "description": "Abrupt emotion change in same session"},
    "extended_session": {"weight": 0.12, "description": "Abnormally long listening session"},
    "repetitive_listening": {"weight": 0.10, "description": "Same track repeated multiple times"},
    "inappropriate_timing": {"weight": 0.08, "description": "Energetic music at 3 AM"},
    "emotional_rollercoaster": {"weight": 0.07, "description": "Rapid emotion changes"},
    "work_hours_party": {"weight": 0.05, "description": "Party music during work hours"},
    "sleep_time_energy": {"weight": 0.08, "description": "High energy music during sleep hours"},
    "morning_melancholy": {"weight": 0.06, "description": "Sad music early morning"},
    "focus_at_midnight": {"weight": 0.09, "description": "Focus music at midnight"}
}


def get_time_of_day(hour):
    if 5 <= hour < 10:
        return "morning"
    elif 10 <= hour < 15:
        return "afternoon"
    elif 15 <= hour < 20:
        return "evening"
    else:
        return "night"

def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "fall"

def choose_emotion_for_hour(hour, date):
    time_of_day = get_time_of_day(hour)
    day_type = "weekend" if date.weekday() >= 5 else "weekday"
    season = get_season(date)
    
    # Base weights from time preferences
    emotions = list(TIME_PREFERENCES.keys())
    weights = [TIME_PREFERENCES[e][time_of_day] for e in emotions]
    
    # Apply day modifiers
    for i, emotion in enumerate(emotions):
        modifier = DAY_MODIFIERS[day_type].get(emotion, 1.0)
        weights[i] *= modifier
        
        # Seasonal adjustments
        if season == "summer":
            if emotion in ["happy", "party", "energetic"]:
                weights[i] *= 1.3
            elif emotion in ["sad", "calm"]:
                weights[i] *= 0.7
        elif season == "winter":
            if emotion in ["sad", "chill", "romantic"]:
                weights[i] *= 1.2
            elif emotion in ["party"]:
                weights[i] *= 0.6
        elif season == "spring":
            if emotion in ["happy", "romantic"]:
                weights[i] *= 1.2
    
    # Normalize
    total = sum(weights)
    if total > 0:
        weights = [w/total for w in weights]
    else:
        weights = [1/len(emotions)] * len(emotions)
    
    return np.random.choice(emotions, p=weights)

def add_seasonal_trends(events):
    """Add seasonal patterns to listening behavior"""
    seasonal_events = []
    for event in events:
        dt = datetime.strptime(event["datetime"], "%Y-%m-%d %H:%M:%S")
        season = get_season(dt)
        
        if season == "summer":
            # More outdoor/party music during summer
            if random.random() < 0.3 and event.get("emotion") in ["party", "happy"]:
                event["sub_text"] = f"Summer Vibes - {event['sub_text']}"
        elif season == "winter":
            # More cozy/chill music during winter
            if random.random() < 0.25 and event.get("emotion") in ["chill", "calm"]:
                event["sub_text"] = f"Cozy Winter - {event['sub_text']}"
        
        seasonal_events.append(event)
    return seasonal_events


def add_enhanced_anomalies(events, anomaly_rate=0.15):
    """Add sophisticated anomalies to listening data"""
    
    print(f"\nAdding enhanced anomalies to {anomaly_rate*100:.0f}% of events...")
    
    events_with_anomalies = []
    anomaly_types_list = list(ANOMALY_TYPES.keys())
    anomaly_weights = [ANOMALY_TYPES[t]["weight"] for t in anomaly_types_list]
    
    # Track repetition counter
    recent_tracks = {}
    
    for i, event in enumerate(events):
        # Track repetition detection
        track_key = f"{event['title']}_{event['text']}"
        recent_tracks[track_key] = recent_tracks.get(track_key, 0) + 1
        
        # Decide if this event should be anomalous
        if random.random() < anomaly_rate:
            anomaly_type = np.random.choice(anomaly_types_list, p=anomaly_weights)
            event = event.copy()
            event["anomaly"] = True
            event["anomaly_type"] = anomaly_type
            event["anomaly_description"] = ANOMALY_TYPES[anomaly_type]["description"]
            
            dt = datetime.strptime(event["datetime"], "%Y-%m-%d %H:%M:%S")
            
            if anomaly_type == "late_night_sad":
                # Change to late night hour and sad emotion
                event["hour"] = random.choice([1, 2, 3, 4])
                event["time_of_day"] = "night"
                event["emotion"] = "sad"
                dt = dt.replace(hour=event["hour"])
                event["timestamp"] = int(dt.timestamp() * 1000)
                event["datetime"] = dt.strftime("%Y-%m-%d %H:%M:%S")
                
                # Pick a sad track
                if "sad" in tracks_by_emotion and tracks_by_emotion["sad"]:
                    sad_track = random.choice(tracks_by_emotion["sad"])
                    event["title"] = sad_track["track_name"]
                    event["text"] = sad_track["artist_name"]
                    event["track_info"]["artist"] = sad_track["artist_name"]
                    event["track_info"]["track"] = sad_track["track_name"]
                    event["sub_text"] = random.choice(PLAYLISTS_BY_EMOTION["sad"])
            
            elif anomaly_type == "sudden_emotion_shift":
                # Shift to contrasting emotion
                current_emotion = event["emotion"]
                contrasting = {
                    "happy": "sad", "sad": "happy", "chill": "energetic",
                    "energetic": "chill", "calm": "party", "party": "calm",
                    "romantic": "energetic", "focus": "party"
                }
                new_emotion = contrasting.get(current_emotion, "sad")
                event["emotion"] = new_emotion
                
                # Pick track with new emotion
                if new_emotion in tracks_by_emotion and tracks_by_emotion[new_emotion]:
                    new_track = random.choice(tracks_by_emotion[new_emotion])
                    event["title"] = new_track["track_name"]
                    event["text"] = new_track["artist_name"]
                    event["track_info"]["artist"] = new_track["artist_name"]
                    event["track_info"]["track"] = new_track["track_name"]
                    event["sub_text"] = random.choice(PLAYLISTS_BY_EMOTION[new_emotion])
            
            elif anomaly_type == "extended_session":
                # Extend duration (handled by session length in generation)
                event["is_extended"] = True
                event["duration_multiplier"] = random.uniform(2.5, 4.0)
            
            elif anomaly_type == "repetitive_listening":
                # Repeat the same track multiple times
                if i > 0:
                    prev_event = events[i-1]
                    event["title"] = prev_event["title"]
                    event["text"] = prev_event["text"]
                    event["track_info"]["artist"] = prev_event["track_info"]["artist"]
                    event["track_info"]["track"] = prev_event["track_info"]["track"]
                    event["is_repetition"] = True
            
            elif anomaly_type == "inappropriate_timing":
                # Energetic music at 3 AM
                event["hour"] = random.choice([2, 3, 4])
                event["time_of_day"] = "night"
                event["emotion"] = "energetic"
                dt = dt.replace(hour=event["hour"])
                event["timestamp"] = int(dt.timestamp() * 1000)
                event["datetime"] = dt.strftime("%Y-%m-%d %H:%M:%S")
                
                if "energetic" in tracks_by_emotion:
                    energetic_track = random.choice(tracks_by_emotion["energetic"])
                    event["title"] = energetic_track["track_name"]
                    event["text"] = energetic_track["artist_name"]
                    event["track_info"]["artist"] = energetic_track["artist_name"]
                    event["track_info"]["track"] = energetic_track["track_name"]
            
            elif anomaly_type == "emotional_rollercoaster":
                # Rapid emotion change
                all_emotions = list(tracks_by_emotion.keys())
                event["emotion"] = random.choice(all_emotions)
                if event["emotion"] in tracks_by_emotion:
                    new_track = random.choice(tracks_by_emotion[event["emotion"]])
                    event["title"] = new_track["track_name"]
                    event["text"] = new_track["artist_name"]
                    event["track_info"]["artist"] = new_track["artist_name"]
                    event["track_info"]["track"] = new_track["track_name"]
            
            elif anomaly_type == "work_hours_party":
                # Party music during work hours
                event["hour"] = random.choice([10, 11, 14, 15])
                event["time_of_day"] = "afternoon"
                event["emotion"] = "party"
                dt = dt.replace(hour=event["hour"])
                event["timestamp"] = int(dt.timestamp() * 1000)
                event["datetime"] = dt.strftime("%Y-%m-%d %H:%M:%S")
                
                if "party" in tracks_by_emotion:
                    party_track = random.choice(tracks_by_emotion["party"])
                    event["title"] = party_track["track_name"]
                    event["text"] = party_track["artist_name"]
                    event["track_info"]["artist"] = party_track["artist_name"]
                    event["track_info"]["track"] = party_track["track_name"]
            
            elif anomaly_type == "sleep_time_energy":
                # High energy during sleep hours
                event["hour"] = random.choice([0, 1, 2])
                event["emotion"] = "energetic"
                dt = dt.replace(hour=event["hour"])
                event["timestamp"] = int(dt.timestamp() * 1000)
                event["datetime"] = dt.strftime("%Y-%m-%d %H:%M:%S")
            
            elif anomaly_type == "morning_melancholy":
                # Sad music early morning
                event["hour"] = random.choice([6, 7])
                event["emotion"] = "sad"
                dt = dt.replace(hour=event["hour"])
                event["timestamp"] = int(dt.timestamp() * 1000)
                event["datetime"] = dt.strftime("%Y-%m-%d %H:%M:%S")
            
            elif anomaly_type == "focus_at_midnight":
                # Focus music at midnight
                event["hour"] = random.choice([23, 0, 1])
                event["emotion"] = "focus"
                dt = dt.replace(hour=event["hour"])
                event["timestamp"] = int(dt.timestamp() * 1000)
                event["datetime"] = dt.strftime("%Y-%m-%d %H:%M:%S")
        
        else:
            event["anomaly"] = False
            event["anomaly_type"] = None
        
        events_with_anomalies.append(event)
    
    return events_with_anomalies


def generate_enhanced_listening_data(num_days=365, sessions_per_day=(3, 15), tracks_per_session=(4, 30)):
    """Generate massive synthetic listening data with enhanced realism"""
    
    events = []
    start_date = datetime(2025, 1, 1)
    
    music_apps = [
        ("Spotify", "com.spotify.music"),
        ("YouTube Music", "com.google.android.apps.youtube.music"),
        ("Apple Music", "com.apple.android.music"),
        ("Amazon Music", "com.amazon.music"),
        ("Tidal", "com.tidal.music")
    ]
    
    total_events = 0
    expected_events = num_days * np.mean(sessions_per_day) * np.mean(tracks_per_session)
    print(f"\nGenerating {num_days:,} days of enhanced listening data...")
    print(f"   Expected events: ~{int(expected_events):,}")
    
    for day in tqdm(range(num_days), desc="Generating days"):
        current_date = start_date + timedelta(days=day)
        
        # More sessions on weekends
        is_weekend = current_date.weekday() >= 5
        sessions_min, sessions_max = sessions_per_day
        if is_weekend:
            sessions_min = int(sessions_min * 1.5)
            sessions_max = int(sessions_max * 1.3)
        
        num_sessions = random.randint(sessions_min, sessions_max)
        
        for session_num in range(num_sessions):
            # Session start time with realistic distribution
            hour_weights = [0.08, 0.12, 0.15, 0.12, 0.10, 0.15, 0.12, 0.08, 0.05, 0.03]
            hours = [7, 9, 12, 15, 18, 20, 21, 22, 23, 0]
            
            hour = random.choices(hours, weights=hour_weights)[0]
            minute = random.randint(0, 59)
            second = random.randint(0, 59)
            
            # Adjust date for sessions after midnight
            session_date = current_date
            if hour < 6:
                session_date = current_date + timedelta(days=1)
            
            current_time = datetime(
                session_date.year, session_date.month, session_date.day,
                hour, minute, second
            )
            
            # Choose app and playlist
            app_name, package_name = random.choice(music_apps)
            
            # Session emotion (can evolve during session)
            session_emotion = choose_emotion_for_hour(current_time.hour, current_date)
            
            # Number of tracks in session
            num_tracks = random.randint(*tracks_per_session)
            
            # Track repetition counter for this session
            session_tracks = []
            
            for track_num in range(num_tracks):
                # Emotion can shift within session
                if track_num > 0 and random.random() < 0.15:
                    session_emotion = choose_emotion_for_hour(current_time.hour, current_date)
                
                # Get track for current emotion
                if session_emotion in tracks_by_emotion and tracks_by_emotion[session_emotion]:
                    track = random.choice(tracks_by_emotion[session_emotion])
                else:
                    fallback = list(tracks_by_emotion.keys())[0]
                    track = random.choice(tracks_by_emotion[fallback])
                
                # Choose playlist name
                playlist_name = random.choice(PLAYLISTS_BY_EMOTION.get(session_emotion, ["My Playlist"]))
                
                # Create event in exact format
                event = {
                    "timestamp": int(current_time.timestamp() * 1000),
                    "datetime": current_time.strftime("%Y-%m-%d %H:%M:%S"),
                    "package": package_name,
                    "app": app_name,
                    "title": track.get("track_name", "Unknown"),
                    "text": track.get("artist_name", "Unknown Artist"),
                    "big_text": "",
                    "sub_text": playlist_name,
                    "is_playing": False,
                    "track_info": {
                        "artist": track.get("artist_name", "Unknown"),
                        "track": track.get("track_name", "Unknown"),
                        "state": "unknown"
                    },
                    "emotion": session_emotion,
                    "hour": current_time.hour,
                    "time_of_day": get_time_of_day(current_time.hour),
                    "day_of_week": current_date.weekday(),
                    "is_weekend": is_weekend,
                    "season": get_season(current_date)
                }
                
                events.append(event)
                total_events += 1
                session_tracks.append(event)
                
                # Move time forward
                duration = track.get("duration_ms", random.randint(180000, 300000))
                gap = random.randint(0, 8000)
                current_time += timedelta(milliseconds=duration + gap)
            
            # Session gap
            session_gap = random.randint(300000, 3600000)
            current_time += timedelta(milliseconds=session_gap)
    
    return events



# Generate 1 year of data
NUM_DAYS = 365
listening_data = generate_enhanced_listening_data(
    num_days=NUM_DAYS,
    sessions_per_day=(4, 20),
    tracks_per_session=(5, 35)
)

print(f"\nGenerated {len(listening_data):,} listening events")

# Add seasonal trends
listening_data = add_seasonal_trends(listening_data)

# Add enhanced anomalies
listening_data = add_enhanced_anomalies(listening_data, anomaly_rate=0.15)

# Save to JSON
output_file = "music_listening_data.json"
with open(output_file, "w") as f:
    json.dump(listening_data, f, indent=2)

print(f"\n Saved {len(listening_data):,} events to {output_file}")
print(f"File size: {len(json.dumps(listening_data)) / (1024*1024):.2f} MB")


df = pd.DataFrame(listening_data)
df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='ms')

print(f"\nOverall Statistics:")
print(f"   Total events: {len(df):,}")
print(f"   Unique tracks: {df['title'].nunique():,}")
print(f"   Unique artists: {df['text'].nunique():,}")
print(f"   Days covered: {df['timestamp_dt'].dt.date.nunique()}")
print(f"   Months covered: {df['timestamp_dt'].dt.month.nunique()}")

print(f"\nEmotion Distribution:")
emotion_counts = df['emotion'].value_counts()
for emotion, count in emotion_counts.items():
    print(f"   {emotion:12}: {count:8,} ({count/len(df)*100:5.1f}%)")

print(f"\nAnomaly Distribution:")
anomaly_count = df['anomaly'].sum()
print(f"   Normal: {len(df) - anomaly_count:8,} ({(1 - anomaly_count/len(df))*100:5.1f}%)")
print(f"   Anomaly: {anomaly_count:8,} ({anomaly_count/len(df)*100:5.1f}%)")

if anomaly_count > 0:
    print(f"\n   Anomaly Type Breakdown:")
    anomaly_types = df[df['anomaly'] == True]['anomaly_type'].value_counts()
    for atype, count in anomaly_types.items():
        print(f"     {atype:25}: {count:6,} ({count/anomaly_count*100:5.1f}%)")

print(f"\nTime of Day Distribution:")
time_counts = df['time_of_day'].value_counts()
for time_of_day, count in time_counts.items():
    print(f"   {time_of_day:10}: {count:8,} ({count/len(df)*100:5.1f}%)")

print(f"\n Day Type Distribution:")
weekend_count = df['is_weekend'].sum()
weekday_count = len(df) - weekend_count
print(f"   Weekday: {weekday_count:8,} ({weekday_count/len(df)*100:5.1f}%)")
print(f"   Weekend: {weekend_count:8,} ({weekend_count/len(df)*100:5.1f}%)")

print(f"\n Seasonal Distribution:")
season_counts = df['season'].value_counts()
for season, count in season_counts.items():
    print(f"   {season:10}: {count:8,} ({count/len(df)*100:5.1f}%)")

print(f"\nApp Distribution:")
app_counts = df['app'].value_counts().head()
for app, count in app_counts.items():
    print(f"   {app:15}: {count:8,} ({count/len(df)*100:5.1f}%)")

print(f"\nTop 10 Playlists:")
playlist_counts = df['sub_text'].value_counts().head(10)
for playlist, count in playlist_counts.items():
    print(f"   {playlist[:40]:40}: {count:6,}")

print(f"\nTop 10 Artists:")
artist_counts = df['text'].value_counts().head(10)
for artist, count in artist_counts.items():
    print(f"   {artist[:40]:40}: {count:6,}")

print(f"\nTop 10 Tracks:")
track_counts = df['title'].value_counts().head(10)
for track, count in track_counts.items():
    print(f"   {track[:50]:50}: {count:6,}")


print(f"\nReady for:")
print(f"   • Time-series analysis (Prophet, TBATS, LSTM)")
print(f"   • Advanced anomaly detection with {anomaly_count:,} labeled anomalies")
print(f"   • Emotion prediction with {len(df):,} labeled samples")
print(f"   • Pattern recognition across {NUM_DAYS} days of data")

Loading audio features from audio_features_clean.csv...
Loaded 95,235 tracks with audio features

Tracks available per emotion:
  calm: 12,509 tracks
  party: 10,150 tracks
  energetic: 11,909 tracks
  chill: 13,112 tracks
  happy: 12,196 tracks
  romantic: 12,306 tracks
  focus: 11,313 tracks
  sad: 11,740 tracks


Generating 365 days of enhanced listening data...
   Expected events: ~87,600


Generating days: 100%|██████████| 365/365 [00:03<00:00, 93.97it/s] 



Generated 93,653 listening events

Adding enhanced anomalies to 15% of events...

 Saved 93,653 events to music_listening_data.json
File size: 45.93 MB


Overall Statistics:
   Total events: 93,653
   Unique tracks: 48,909
   Unique artists: 27,760
   Days covered: 367
   Months covered: 12

Emotion Distribution:
   sad         :   18,371 ( 19.6%)
   energetic   :   14,133 ( 15.1%)
   romantic    :   13,378 ( 14.3%)
   party       :   11,397 ( 12.2%)
   focus       :    9,785 ( 10.4%)
   happy       :    9,654 ( 10.3%)
   chill       :    9,337 ( 10.0%)
   calm        :    7,598 (  8.1%)

Anomaly Distribution:
   Normal:   79,535 ( 84.9%)
   Anomaly:   14,118 ( 15.1%)

   Anomaly Type Breakdown:
     late_night_sad           :  2,877 ( 20.4%)
     sudden_emotion_shift     :  2,084 ( 14.8%)
     extended_session         :  1,697 ( 12.0%)
     repetitive_listening     :  1,407 ( 10.0%)
     focus_at_midnight        :  1,237 (  8.8%)
     inappropriate_timing     :  1,178 (  8.3%)
     s